# Optimising the relationship between form similarity and meaning similarity

Kinship terms, the set of words that denote family relationships, are strikingly diverse across the world's languages. Different languages incorporate different aspects of meaning into their kinship terminology. In this project, we explore the extent of this semantic variation to identify whether there are common constraints on the meanings of kin terms, or whether languages are free to vary which semantics they encode with their kinship terminology.

In this notebook, I build a model that quantifies the extent to which each of six semantic features is encoded in the kinship terminology of particular languages. These six features are Generation, Gender, Lineality, Gender of Connecting Relative, Seniority, and Speaker Gender -- all of which may form part of the meaning of a kin term in some language.

Language is topographic, meaning that there is a close mapping between shared meaning and shared form. Words with more similar meanings should have more similar forms -- so if we measure "shared meaning" and "shared form", we should expect them to have a high correlation.

Except not all semantic features are encoded in every language's kinship terminology, nor are they encoded equally within an individual language.

So, in order to determine how much each feature is encoded in a language, we must first assume that kin terms are maximally topographic. We work backwards to work out which features lead to a high correlation between forms and meanings, which features make no difference, and which features hinder this correlation. This will allow us to model which semantic features contribute most to shared form-meaning mappings, and which contribute less, allowing us to model cross-linguistic variation on kin term semantics.

To do this, the model optimises the correlation between two metrics: the semantic similarity shared between kin types, and the form similarity shared between kin terms.

To do this, we need to:

* Define a measure of semantic similarity between kin terms.
* Define a measure of form similarity between kin terms.
* Optimise the semantic similarity measure for the highest possible correlation.

First, import all necessary packages.

In [7]:
import csv # for writing dataframes to CSV

import kintypes as kt # a list of kin types to extract from the kinbank data

# for data wrangling
import pandas as pd 
import numpy as np
import math
import statistics
import matplotlib.pyplot as plt

# for running the optimisation algorithm
from scipy.optimize import minimize
from scipy.stats import pearsonr
from scipy.optimize import Bounds

# for calculating model runtime
from tqdm import tqdm
import time
import timeit
import datetime

import itertools as it # iterating random sets of weights for initialising the model
from Levenshtein import distance as lvs_dist # calculating levenshtein distance
import random

Next, read in the Kinbank data -- a CSV file containing data on kinship terminology from 1200+ languages (Passmore et al., 2023).

In [9]:
kinbank = pd.read_csv('kinbank.csv')

The following function extracts the relevant data from the `kinbank` dataframe.

For a particular language (specified by its `glottocode`), and for each kin type listed in `kt.relatives` (a list of 100 kin types imported from a separate file), create a key:value pair where the key is the kin type (i.e. the precise meaning of the kin term) and the value is the kin term (i.e. the wordform used in a particular language to refer to that kin type).

Return a dictionary of all key:value pairs.

In [10]:
def get_kin_terms(glottocode):
    language_data = kinbank[kinbank['Glottocode'] == glottocode] # get the kin terms for 
                                                                 # the language specified by the glottocode

    kinterms = {}

    for relative in kt.relatives: # iterate over all 100 kin types
        row = language_data[language_data['Parameter_ID'] == relative] # grab the data for this kin type for this language
        if row.empty: # if there is no data on this kin type, do nothing
            pass
        else: # if there is
            term = list(row['Form'])[-1] # get the kin term used for this kin type
            # print(relative,term)
            kinterms[relative] = term # and make a key:value pair of kin type:kin term

    return kinterms

Create a list of all glottocodes so we can iterate over all languages later.

In [11]:
all_glottocodes = list(set(kinbank['Glottocode']))

# Calculating semantic similarity

The functions in the following cell take a single argument, `relative`, which should correspond to a kin type listed in `kt.relatives`.

The functions return a kin type's value for one of six semantic features: 
- Generation (the generation a relative belongs to, e.g. 'grandparental',
- Gender (the gender of the relative, e.g. man or woman),
- Lineality (is this relative in your direct bloodline),
- Gender of Connecting Relative (is this relative connected to you by a man or a woman, e.g. mother's side vs father's side),
- Seniority (is this relative older or younger than their counterpart),
- and Speaker Gender (what is the gender of the person using the kin term)

In Kinbank, kin types are given in shorthand, e.g. `mMeZ` for 'man's mother's elder sister': `m` for 'male speaker', `M` for 'mother', `e` for 'elder' and `Z` for 'sister'. These shorthand codes indicate parts of their semantics, e.g. we can know that all women will end in `M` for mother, `Z` for sister, or `D` for daughter.


In [12]:
# GENERATION
def determine_generation(relative):
    
    # `kt.gN` are hand-curated lists of which generation each kin type belongs to - these lists are in kintypes.py
    # There is no way to automatically assign generation based on the kin type shorthand.
    
    if relative in kt.g5:
        return 2
    if relative in kt.g4:
        return 1
    if relative in kt.g3:
        return 0
    if relative in kt.g2:
        return -1
    if relative in kt.g1:
        return -2
    

# GENDER
def determine_gender(relative):
    
    women = ['M','Z','D'] # mothers, sisters, daughters
    men = ['F','B','S'] # fathers, brothers, son

    final_letter = relative[-1] # the final letter of `relative` will always be one of the above letters
    
    if final_letter in women:
        gender = 'f'
    elif final_letter in men:
        gender = 'm'
        
    return gender


# LINEALITY
def determine_line(relative):
    
    lineage = [relative[0] + x for x in ['M','F','MM','FF','B','Z','S','D','SS','DD']] 
    # `lineage` is a list of all relatives in the direct line (with relative[0] ('m' or 'f') added to indicate speaker gender)
    
    # remove any indication of relative age from the kin type, for ease
    relative = relative.replace('e','')
    relative = relative.replace('y','')
    
    if relative in lineage:
        line = 'l' # lineal
    else:
        line = 'c' # collateral
       
    return line


# GENDER OF CONNECTING RELATIVE
def determine_gcr(relative):
    
    # remove any indication of relative age from the kin type
    relative = relative.replace('e','')
    relative = relative.replace('y','')
    
    women = ['M','Z','D'] # mothers, sisters, daughters
    men = ['F','B','S'] # fathers, brothers, sons
    
    na = ['mB','mZ','fB','fZ','mD','mS','fD','fS'] # siblings and children are not connected by another relative, so they receive NA values
    
    if relative in na:
        gcr = 'na'
    else:
        if relative[1] in women:
            gcr = 'f' 
        elif relative[1] in men:
            gcr = 'm'
    
    return gcr


# SENIORITY
def determine_seniority(relative):
    
    if 'e' in relative: # e = elder
        return 'e'
    elif 'y' in relative: # y = younger
        return 'y'
    else:
        return 'na'

    
# SPEAKER GENDER
def determine_speaker(relative):
    
    speaker = relative[0] # relative[0] = m or f, indicates speaker gender
        
    return speaker

The following function, `semantic_similarity`, takes two kin types `x` and `y`, and determines each of their values on each feature. The function compares their values for each feature and assigns 1 if they share a value, -1 if they do not share a value, and 0 if one of them takes an NA value. It builds a matrix which determines which features are shared and which are not, and multiplies that matrix by a set of `weights` on each feature (as default, all features are weighted equally). 

It returns the sum of the matrix.

In [13]:
def semantic_similarity(x,y,weights = [1/6]*6):
    
    sem_distance = [1 for x in weights] # set up the semantic distance matrix
    
    if determine_generation(x) != determine_generation(y):
        sem_distance[0] = -1
                
    if determine_gender(x) != determine_gender(y):
        sem_distance[1] = -1
        
    if determine_line(x) != determine_line(y):
        sem_distance[2] = -1
        
    if determine_gcr(x) == 'na' or determine_gcr(y) == 'na':
        sem_distance[3] = 0 # if either kin type has an NA value for this feature, assign 0
    elif determine_gcr(x) != determine_gcr(y):
        sem_distance[3] = -1
        
    if determine_seniority(x) == 'na' or determine_seniority(y) == 'na':
        sem_distance[4] = 0 # if either kin type has an NA value for this feature, assign 0
    elif determine_seniority(x) != determine_seniority(y):
        sem_distance[4] = -1
        
    if determine_speaker(x) != determine_speaker(y):
        sem_distance[5] = -1
    
    weighted_sem_distance = [x*y for x,y in zip(sem_distance,weights)]
    
    return sum(weighted_sem_distance)

For example, the similarity between 'male speaker's elder brother' and male speaker's elder sister' is 0.5 -- they share Generation, Lineality, Seniority, Speaker Gender (sum to 4), do not share Gender (subtract 1), and have NA for GCR (add 0). When treating all features equally, this is equivalent to sharing 3/6 features or having 0.5 similarity.

In [14]:

semantic_similarity('meB','meZ')

0.5

# Measuring form similarity

Form similarity is measured as inverse normalised Levenshtein distance, the number of insertions, deletions or substitutions required to turn one string into another.

The following function takes two kin types `x` and `y`, and returns 1 - their normalised Levenshtein edit distance.

In [15]:
def edit_similarity(x,y):
    return 1 - (lvs_dist(x,y) / max(len(x),len(y)))

In [16]:
# for example

edit_similarity('grandmother','grandson')

0.5454545454545454

# Optimise the semantic similarity metric

Now we have our metrics, we will build a model that optimises the correlation between them by manipulating the weight assigned to each semantic feature. By taking the weights that provide the greatest correlation, we can measure which features contribute most to shared form-meaning mappings in each language -- higher weighted features contribute most, lower weighted features contribute least. We will use a gradient descent algorithm to achieve this.

The following function defines the objective function for our gradient descent algorithm: Pearson correlation between semantic similarity and edit similarity.

It takes two arguments: `params`, a set of weights on each feature, and `ks`, the kinship system of a particular language.
It calculates the matrix of semantic similarities and form similarities over all pairwise kin types in `ks` (weighted by `params`) and the edit similarity between all the terms in `ks`.

It returns the inverse correlation between these two matrices (because we want gradient descent to output the maximal correlation, not the minimal).


In [17]:
def objective_function(params,ks):
    
    semantic_distance = []
    edit_distance = []
    
    for relative1 in ks: # for each relative
        for relative2 in ks: # and each other relative
            if relative1 != relative2: # if they are not the same
                semantic_distance.append(semantic_similarity(relative1,relative2,params)) # 
                edit_distance.append(edit_similarity(ks[relative1],ks[relative2]))
        
    
    correlation = pearsonr(edit_distance,semantic_distance)[0]
    
#     save_data(language,code,normalise(params),correlation)
    
    end = datetime.datetime.now()
    print('Iteration completed at', end.time())
    
    return -correlation  # negative correlation, because we want the maximum maximization

To run the optimisation, we need a way to initialise a random set of parameter weights. 

Parameter weights should be a list of six values between 0 and 1, that also sum to 1.

The following cell sets possible values for each weight based on two arguments, `increments` and `num_parameters` (in our case, six).

By permutation, it creates a list of all possible sets of initialising weights.

We then define `possible_weights` - a list of all possible sets of weights, from which we will select a random one to initialise the model.

In [18]:
def permute(increments,num_parameters):
    
    # all possible numbers between 0 and 1 in increments specified by `increments`
    weight_values = [round(value,2) for value in (list(it.islice(it.count(0, increments), int((1/increments)+1))))]
    
    # take weight_values, and output every set of six that sum to 1
    possible_weights = [weights for weights in it.combinations_with_replacement(weight_values,num_parameters) if sum(weights) == 1]

    all_permutations = []

    # permute possible_weights, such that any value can appear in any index    
    for weight in possible_weights:
        permutations = it.permutations(weight)
        for permutation in set(permutations):
            all_permutations.append(list(permutation))
            
    return all_permutations


possible_weights = permute(0.05,6)

We want the model to only generate weights that sum to 1, so we have to create a function to normalise the set of weights:

In [19]:
def normalise(arr):
    return(arr/sum(arr))

We want to save the optimisation output to CSV, so let's specify what data we want to save.

The function `save_result` takes three arguments: the `language` currently being optimised, the `code` (glottocode) of that language, and the `result` - the optimisation algorithm's solution at a particular point in time.

In [20]:
def save_result(language,code,result):
    data = {}
    
    global df
    global simulation
    
    params = normalise(result.x) # result.x is the set of parameters the model is currently using
    corr = -result.fun # result.fun is the correlation, take the inverse to make it positive
    
    data['language'] = language
    data['code'] = code
    data['generation'] = params[0] # the index of each parameter corresponds to the weight on that feature
    data['gender'] = params[1]
    data['lineality'] = params[2]
    data['gcr'] = params[3]
    data['seniority'] = params[4]
    data['speaker_gender'] = params[5]
    data['result'] = corr
    
    df.append(data)

### Finally, run the optimisation algorithm.

Note: this takes several hours. For testing purposes, comment out `for code in all_glottocodes` so you can run it on a single language, which is much faster!

In [22]:
# set the start time so we can see how long the model took to run
start = datetime.datetime.now()
print('Optimization started at ', start.time())

# set the bounds for each parameter between 0 and 1
bounds = [(0,1),(0,1),(0,1),(0,1),(0,1),(0,1)]

# create an empty df for the data to be appended to
df = []

# set the output filename
filename = 'model_run'

for code in ['stan1293']: # uncomment this line to run only on English

# for code in all_glottocodes: # for all languages in kinbank - comment out this line to run only on English
    ks = get_kin_terms(code) # get the kinship system
    language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name

    initial_guess = random.choice(possible_weights) # choose a random initial set of parameters
    
    print('initial_guess:', initial_guess)

    # run the optimisation algorithm with method set to `l-bfgs-b` and bounds set to (0,1) for each parameter
    # args = ks allows the kinship system to be passed to the objective function (params is passed automatically)
    result = minimize(objective_function, 
                      initial_guess,
                      args = (ks),
                      method = 'L-BFGS-B',
                      bounds = bounds)
        
    # for each language, store the finished optimisation result to a df
    save_result(language,code,result) # save the language, the parameters, and the result

    # write that data to csv
    pd.DataFrame(df).to_csv('../analysis/data/' + filename + '.csv',index=False,mode = 'w')

# set the end time
end = datetime.datetime.now()
print('Optimization ended at ', end.time())

# print the dataframe
pd.DataFrame(df)

Optimization started at  08:40:20.637154
initial_guess: [0.25, 0.3, 0, 0.25, 0.1, 0.1]
Iteration completed at 08:40:20.890955
Iteration completed at 08:40:21.018087
Iteration completed at 08:40:21.193912
Iteration completed at 08:40:21.335968
Iteration completed at 08:40:21.464932
Iteration completed at 08:40:21.639858
Iteration completed at 08:40:21.799037
Iteration completed at 08:40:22.005302
Iteration completed at 08:40:22.185387
Iteration completed at 08:40:22.304668
Iteration completed at 08:40:22.464117
Iteration completed at 08:40:22.624340
Iteration completed at 08:40:22.832525
Iteration completed at 08:40:22.944097
Iteration completed at 08:40:23.134670
Iteration completed at 08:40:23.279109
Iteration completed at 08:40:23.454252
Iteration completed at 08:40:23.594353
Iteration completed at 08:40:23.776135
Iteration completed at 08:40:23.946282
Iteration completed at 08:40:24.115936
Iteration completed at 08:40:24.289418
Iteration completed at 08:40:24.423318
Iteration comple

,language,code,generation,gender,lineality,gcr,seniority,speaker_gender,result
0,English,stan1293,0.731235,0.072595,0.19617,0.0,0.0,0.0,0.665784


The algorithm produces a parameter value for each feature -- this value can be thought of as **the feature's contribution to form-meaning mappings**, i.e. to what extent is this feature useful for determining whether two kin types share a kin term in this language.

For example, the output parameters for English are (rounded to 2 dp):
* Generation 0.73
* Gender 0.07
* Lineality 0.20
* GCR 0
* Seniority 0
* Speaker Gender 0

English makes no distinctions between kin on the basis of GCR, Seniority, or Speaker Gender -- so as expected, we get 0 values for these features.

The model output also tells us that Generation is by far the most important indicator of whether two kin share part of their form (e.g. 'aunt' refers to mother's sister AND father's sister); in fact kin in different generations never share a term. Gender is also relevant, but much less so -- many kin share Gender but no part of their form (e.g. 'mother' and 'aunt'). Lineality is relevant as kin in the direct line often share part of their form (e.g. the 'grand' in 'grandmother' and 'granddaughter').

By looking at these values for each language in the dataset, and comparing values across languages, we can infer constraints on the meanings of kin terms. (All further analysis is in a separate file, conducted in R rather than Python).

# Appendix B

The results of the optimisation algorithm suggest that semantic features are encoded hierarchically in kinship systems across the world's languages. 

To show that this result is robust to permutation, we scramble which terms correspond to which types within a language, then run the gradient descent algorithm.

By permuting which kin types correspond to which kin terms, we break any topographic relationship between form and meaning.

If our results are not actually to do with topographic mappings between forms and meanings, then we should still get the same distribution of weights on each feature after permutation (but we don't!)


First, we write a function `shuffle_ks`, which takes a single argument `ks` - the kinship system of a particular language.
It then shuffles which kin terms map to which kin types, and returns this new permuted kinship system:

In [35]:
def shuffle_language(ks):
    new_ks = {}
    
    terms = list(ks.values()) # get the kin terms of this language
    relatives = list(ks.keys()) # get the kin types of this language
    
    random.shuffle(terms) # randomly shuffle the order of `terms`
    
    # and then reassign the terms to types in this random order
    for i in range(len(ks)):
        new_ks[relatives[i]] = terms[i]
        
    return new_ks

Now we can run the optimisation again, but this time, we'll shuffle the kinship systems before we optimise them.

We'll also permute each language a certain number of `times` e.g. 1000 times, to ensure that these permuted results are significant different from the veridical results. 

**WARNING: this takes multiple days to run!** If you'd like to test it, set `times` to 1.

In [39]:
# set the start time
start = datetime.datetime.now()
print('Optimization started at ', start.time())

# set the number of iterations
times = 1000

# set the bounds for each parameter between 0 and 1
bounds = [(0,1),(0,1),(0,1),(0,1),(0,1),(0,1)]

# create an empty df for the data to be appended to
df = []

# set the output filename
filename = 'model_baseline'

for code in all_glottocodes: # for every language in Kinbank
    ks = get_kin_terms(code) # get that language's kinship system
    language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
    
    for i in range(times):
        
        simulation = str(i) # assign an ID to this simulation, i.e. the iteration count
        
        shuffled_ks = shuffle_language(ks) # shuffle the veridical kinship system

        data = {} # create an empty df

        initial_guess = random.choice(possible_weights) # choose a random initial set of parameters
        print('initial_guess = ', initial_guess)
    
        # run the gradient descent algorithm
        result = minimize(objective_function, 
                              initial_guess,
                              args = (shuffled_ks),
                              method = 'L-BFGS-B',
                              bounds = bounds)

        save_result(language,code,result) # save the language, the parameters, and the result
    
    # save the data to csv
    pd.DataFrame(df).to_csv('../analysis/data/' + filename + '.csv',index=False,mode = 'w')
    
# set the end time
end = datetime.datetime.now()
print('Optimization ended at ', end.time())

# print the dataframe
pd.DataFrame(df)

Optimization started at  15:46:59.428664
initial_guess =  [0.05, 0.25, 0.05, 0.1, 0.25, 0.3]
Iteration completed at 15:46:59.732415
Iteration completed at 15:46:59.950572
Iteration completed at 15:47:00.162399
Iteration completed at 15:47:00.345252
Iteration completed at 15:47:00.471972
Iteration completed at 15:47:00.662197
Iteration completed at 15:47:00.853052
Iteration completed at 15:47:01.013454
Iteration completed at 15:47:01.140716
Iteration completed at 15:47:01.354197
Iteration completed at 15:47:01.523477
Iteration completed at 15:47:01.667326
Iteration completed at 15:47:01.826303
Iteration completed at 15:47:01.985688
Iteration completed at 15:47:02.177167
Iteration completed at 15:47:02.384984
Iteration completed at 15:47:02.571174
Iteration completed at 15:47:02.704048
Iteration completed at 15:47:02.830767
Iteration completed at 15:47:02.973883
Iteration completed at 15:47:03.117179
Iteration completed at 15:47:03.261203
Iteration completed at 15:47:03.467169
Iteration 

,language,code,generation,gender,lineality,gcr,seniority,speaker_gender,result
0,English,stan1293,0.000000,0.0,0.0,0.000000,0.00000,1.000000,0.045816
1,Latin,lati1261,0.783122,0.0,0.0,0.004232,0.08197,0.130676,0.037993


# Appendix A

Our model depends on the assumption that kin terms are topographic; i.e. that for each language, there is a strong correlation between our measure of semantic similarity and our measure of form similarity.

This correlation should be stronger after optimisation, reflecting that the model has successfully identified which features contribute most to form-meaning mappings in that language.

Additionally, the correlation after optimisation should be robust to permutation -- otherwise, we cannot trust that the model is actually finding the parameter weights on each feature that contribute most to topographicity.

In this Appendix, we confirm that the optimisation model works as intended, by showing that the optimised correlations are significantly better than we would expect if the model was not actually identifying topographicity.


First, the function `true_corr` calculates the calculation between semantic similarity and form similarity when all features are weighted equally, and saves this data to CSV. 

In [40]:
def true_corr(filename):
    
    df = [] # create an empty dataframe

    for code in tqdm(all_glottocodes): # for every language in kinbank
        data = {}
        
        ks = get_kin_terms(code) # get the kin terms for that language
        
        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the name of the language
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # and get its language family

        semantic_mtx = []
        edit_mtx = []
        
        try:
            # for every pair of relatives in the kinship system, calculate their semantic and edit similarity
            for relative1 in ks:
                for relative2 in ks:
                    if relative1 != relative2:
                        semantic_mtx.append(semantic_similarity(relative1,relative2,[1/6]*6)) 
                        edit_mtx.append(edit_similarity(ks[relative1],ks[relative2]))


            correlation = pearsonr(edit_mtx,semantic_mtx)[0] # calculate the correlation between the matrices

            data['language'] = language
            data['glottocode'] = code
            data['family'] = family
            data['correlation'] = correlation

            df.append(data)
            
        except:
            pass

    # save to CSV
    pd.DataFrame(df).to_csv('../analysis/data/' + filename + '.csv',index=False,mode = 'w')

    # return the dataframe
    return pd.DataFrame(df)
    

In [37]:
true_corr('true_correlations')

100%|██████████████████████████████████████████████████████████████████████████████| 1127/1127 [01:38<00:00, 11.38it/s]


,language,glottocode,family,correlation
0,Southeastern Tepehuan,sout2976,Uto-Aztecan,0.321330
1,Karadjeri,kara1476,Pama-Nyungan,0.249616
2,Comanche,coma1245,Uto-Aztecan,0.317181
3,Arosi,aros1241,Austronesian,0.343972
4,Alamblak,alam1246,Sepik,0.184409
...,...,...,...,...
1118,Bindal-Mount Elliot,bind1236,Pama-Nyungan,0.434719
1119,Ngalkbun,ngal1292,NaN,0.259719
1120,Batanga,bata1285,Atlantic-Congo,0.441348
1121,Luguru,lugu1238,Atlantic-Congo,0.183712


Next, let's ensure that the set of parameter weights output by the optimisation algorithm result in a better correlation for the veridical kinship system compared to permuted versions of the kinship system.

A permuted kinship system has no topographic mapping between forms and meanings (since terms are now randomly assigned to types). If the correlation between form and meaning was to still be very high for these permuted system, this would suggest that our model is not capturing anything about the topographic mappings between forms and meanings.

Let's read in the dataframe containing the optimised weights on each feature for each language:

In [44]:
weights = pd.read_csv('..analysis/data/model_run.csv')

weights

,Glottocode,Name,Family,language,gender,speaker_gender,side,generation,age,lineal,result,simulation
0,abui1241,Abui,NaN,Abui,0.012852,0.000000,0.042729,0.933548,0.010870,0.000000,0.727955,n
1,ache1246,Aché,Tupian,Aché,0.056446,0.035427,0.009476,0.741918,0.000000,0.156733,0.703246,n
2,achi1257,Acehnese,Austronesian,Acehnese,0.216683,0.000000,0.000000,0.553950,0.148505,0.080863,0.597522,n
3,achu1249,Achuar,Chicham,Achuar,0.190981,0.000000,0.018558,0.658717,0.131743,0.000000,0.746811,n
4,adan1251,Adang,NaN,Adang,0.016328,0.000000,0.079821,0.903851,0.000000,0.000000,0.567665,n
...,...,...,...,...,...,...,...,...,...,...,...,...
1222,zigu1242,Zigula,Atlantic-Congo,Zigula,0.043209,0.000000,0.055148,0.656679,0.017510,0.227455,0.536875,n
1223,zinz1238,Zinza,Atlantic-Congo,Zinza,0.168763,0.000000,0.000000,0.791747,0.000000,0.039490,0.663196,n
1224,zulu1248,Zulu,Atlantic-Congo,Zulu,0.079049,0.000000,0.089482,0.684365,0.000000,0.147104,0.395394,n
1225,zulu1248,Zulu,Atlantic-Congo,Zulu,0.079049,0.000000,0.089482,0.684365,0.000000,0.147104,0.395394,n


And then, calculate the correlation between semantic similarity and form similarity for permuted versions of each language.

The function `simulated_corrs` takes two arguments: the output `filename`, and `times` - the number of times to shuffle the kinship system.

For each language (and for `times` number of iterations) it grabs the optimised weights from the `weights` dataframe, and then shuffles that language's kinship system and calculates the correlation between form similarity and semantic similarity for the permuted system.

It then saves this data to CSV. I used it to calculate z-scores for each language, which tell us how many standard deviations each veridical correlation is from the mean of its permuted baseline distribution.

In [64]:
def simulated_corrs(filename,times):
    
    df = []

    for code in tqdm(all_glottocodes): # for every language in Kinbank
        ks = get_kin_terms(code) # get the kin terms for that language
        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the language family
        
        # get the optimal features for that language from the `weights` dataframe
        params = [list(weights[weights['code'] == code]['generation'])[0],
                  list(weights[weights['code'] == code]['gender'])[0],
                  list(weights[weights['code'] == code]['lineal'])[0],
                  list(weights[weights['code'] == code]['gcr'])[0],
                  list(weights[weights['code'] == code]['seniority'])[0],
                  list(weights[weights['code'] == code]['speaker_gender'])[0]]
        
        semantic_mtx = []
        
        # and calculate the semantic similarity between all pairs of kin types in the kinship system
        for relative1 in ks:
            for relative2 in ks:
                if relative1 != relative2:
                    semantic_mtx.append(semantic_similarity(relative1,relative2,params))        
        
        # for a given number of iterations
        for i in range(times):
            
            data = {}

            shuffled_ks = shuffle_language(ks) # shuffle the kinship system
            
            edit_mtx = []

            try:

                # and calculate the edit similarity between all pairs of terms in the shuffled kinship system
                for relative1 in ks:
                    for relative2 in ks:
                        if relative1 != relative2:
                            edit_mtx.append(edit_similarity(shuffled_ks[relative1],shuffled_ks[relative2]))

                # now calculate the correlation between the semantic similarity values, and the edit similarity values
                # (for the shuffled terms)
                correlation = pearsonr(edit_mtx,semantic_mtx)[0]

                data['language'] = language
                data['glottocode'] = code
                data['family'] = family
                data['correlation'] = correlation
                data['sim'] = str(i)

                df.append(data)
            
            except:
                pass

    # save that data to CSV
    pd.DataFrame(df).to_csv('../analysis/data/' + filename + '.csv',index=False,mode = 'w')

    return pd.DataFrame(df)

In [ ]:
simulated_corrs('correlation_baseline')